In [ ]:
# Uncomment if needed to install dependencies in requirements.txt
# %pip install -r ../requirements.txt

# Disclaimer
# This notebook uses of the Physionet Motor Movement/Imagery Dataset.
# Refer to Credits_and_Data in the data folder for details if you want to run it yourself.

In [1]:
import sys
from pathlib import Path
import mne
import numpy as np
mne.set_log_level("ERROR")

In [9]:
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.data import get_subject_data
from src.data import get_labeled_sample_intervals
from src.model import csp_lda_classifier

In [ ]:
# Get all subject folders from Physionet motor movement/imagery dataset
subjects = sorted([child for child in (PROJECT_ROOT/"data"/"files").iterdir() if child.is_dir()])
num_subjects = len(subjects)

window_accuracies = {i: [] for i in range(3, 0, -1)} #Stores accuracies for each subject

# Get results for every subject in physionet dataset
for subject_id, subject in enumerate(subjects):
    subject_id+=1

    print((f"Processing subject {subject_id}/{num_subjects}"))

    # Get runs for subject
    runs = sorted([run for run in subject.glob("*.edf")])
    train_runs = [runs[3], runs[7]]  # Select runs 4, 8 for motor imagery training data
    valid_runs = [runs[11]] # Select run 12 for testing motor imagery model

    # Get labeled sample intervals for simulated online classification of validation data.
    # T1 and T2 as classes to retrieve data for. T1=left hand. T2=right hand.
    labeled_sample_intervals = get_labeled_sample_intervals(valid_runs[0], ("T1","T2"))

    # Load validation data
    raw = mne.io.read_raw_edf(valid_runs[0], preload=True)
    raw.filter(8, 30, fir_design="firwin", phase="zero") # Bandpass filter to focus on relevant frequency range for motor imagery tasks (8-30 Hz)
    data = raw.get_data()
    freq = raw.info["sfreq"] # Get sampling frequency of data

    # Iterate through window sizes of 3, 2, and 1 seconds for classification
    for window_size in range(3, 0, -1):

        # Retrieve EEG data for subject.
        # 1 second offset from event onset to account for subject reaction time to event.
        # T1 and T2 as classes to retrieve data for. T1=left hand. T2=right hand.
        train_data = get_subject_data(train_runs, ("T1","T2"), offset=1, duration=window_size)

        clf = csp_lda_classifier() # Create classifier object

        clf.fit(train_data["x"], train_data["y"]) # Fit classifier with data
    
        step = int(freq*(window_size/4)) # Step size for iterating through validation data
        sample_size = int(freq*window_size) # Interval size for classification

        # Simulate online classification of validation data by iterating through labeled sample intervals
        accuracies = []
        for label, interval in labeled_sample_intervals.items(): # Iterate through intervals
            for event_start, event_end in interval: # 
                for start in range(event_start, event_end - sample_size + 1, step):

                    end = start + sample_size
                    interval = data[:, start:end]

                    predicted = clf.predict(interval[np.newaxis, :, :])[0]
                    accuracies.append(predicted == label)

        mean = np.mean(accuracies)

        print(f"\tWindow size: {window_size} Step size: {window_size/4} Accuracy: {mean}") # Print accuracy for subject
        window_accuracies[window_size].append(mean) # Store accuracy for subject

# Calculate and print overall accuracy across all subjects
window_accuracies = {window_size: np.mean(accuracies) for window_size, accuracies in window_accuracies.items()}
print(f"Overall accuracy across all subjects: \n")
for window_size, accuracy in window_accuracies.items():
    print(f"Window size: {window_size} seconds" 
          f"Step size: {window_size/4} seconds" 
          f"Accuracy: {accuracy}")




Processing subject 1/109
	Window size: 3 Step size: 0.75 Accuracy: 0.7333333333333333
	Window size: 2 Step size: 0.5 Accuracy: 0.5333333333333333
	Window size: 1 Step size: 0.25 Accuracy: 0.593939393939394
Processing subject 2/109
	Window size: 3 Step size: 0.75 Accuracy: 0.8
	Window size: 2 Step size: 0.5 Accuracy: 0.7166666666666667
	Window size: 1 Step size: 0.25 Accuracy: 0.7757575757575758
Processing subject 3/109
	Window size: 3 Step size: 0.75 Accuracy: 0.4666666666666667
	Window size: 2 Step size: 0.5 Accuracy: 0.4666666666666667
	Window size: 1 Step size: 0.25 Accuracy: 0.5636363636363636
Processing subject 4/109
	Window size: 3 Step size: 0.75 Accuracy: 0.6
	Window size: 2 Step size: 0.5 Accuracy: 0.43333333333333335
	Window size: 1 Step size: 0.25 Accuracy: 0.5151515151515151
Processing subject 5/109
	Window size: 3 Step size: 0.75 Accuracy: 0.3333333333333333
	Window size: 2 Step size: 0.5 Accuracy: 0.5666666666666667
	Window size: 1 Step size: 0.25 Accuracy: 0.551515151515